# 第13讲：回到原始会话，核对到底做过什么

总结说‘已经回滚’，原始回执却可能只说明‘收到请求’，甚至是‘权限不足’。本练习先让真实前台留下 memory 和支付操作记录，再让新会话按需回查。Session 保存经过；Memory 保存长期约定，后者不能代替前者证明过去。

本版不重复完整的后台记忆迁移故事。原生 bg-review 不持久化到 SessionDB；我们另设允许 memory 的前台会话，不把观察器 JSON 当作原始会话。

## 准备环境与运行

下面会调用 GLM 并产生 API 费用。按 README 准备环境和凭据，在启动内核前设置 `SESSION_LOOKUP_ALLOW_LIVE=1`，再逐格执行。不设置时第一格即停止，不读取个人凭据、不启服务。无模型检查另用 `bash run.sh self-check`。旧回放缺少前台 memory 会话，当前不可用。

Notebook 直接调用原生 SessionDB 与 session_search。HTTP 服务内的真实 `AIAgent.run_conversation()` 使用 Hermes 自身解释器；没有隐藏的分幕 worker，也不需要再起模型代理。单元格异常或普通中断会等待本次任务结束再清理；强杀内核仍须核查服务状态。

课后系统设计题要求在线巡检 Agent 按需自主回查并纠正总结。事故与新核查会话演示这条主链；前台 memory 会话是附加观察。当前串行队列不验证后台核查与告警处理的调度隔离，回复中的修订也不会自动写入 Memory 或 Skill。请自行分析这些设计取舍，不把程序检查当作系统设计的标准答案。

In [ ]:
from pathlib import Path
import json, os, sys
HERE = Path.cwd()
if not (HERE / 'demo.py').exists():
    HERE = HERE / 'examples/13-session-lookup'
sys.path.insert(0, str(HERE.resolve()))
from notebook_workshop import NotebookWorkshop
workshop = NotebookWorkshop(get_ipython())
run = workshop.prepare()
from common import api
from demo import MEMORY_REQUEST, INCIDENT_REQUEST, AUDIT_REQUEST
import lab
from hermes_state import SessionDB
from tools.session_search_tool import session_search
from validation_checks import pair_messages, execution_state, digest
tasks = {}


## 留下一次真实前台 memory 调用

这个会话只允许 memory。消息要求保存一条教学约定；工具参数、调用编号和工具回执由 Hermes 原生执行循环持久化。助手说‘已保存’不算检查结果，后面还要从 SessionDB 展开核对。

In [ ]:
memory_sid = 'memory-' + workshop.suffix
api('/sessions', {'id': memory_sid, 'title': '前台记忆工具留痕', 'memory_demo': True})
print(MEMORY_REQUEST)
memory_tid = lab.send(memory_sid, MEMORY_REQUEST)
tasks['memory'], _ = lab.wait(memory_tid)


## 留下实际操作与业务复查

支付现场全部是构造数据。当前身份提交回滚会被平台拒绝；用户另外授权暂停 repeat-pay。Agent 须重新查部署与日志，不能用操作成功回执代替业务状态，也不能仅凭时间先后认定恢复原因。

In [ ]:
incident_sid = 'incident-' + workshop.suffix
api('/sessions', {'id': incident_sid, 'title': '支付事故调查 rollback'})
print(INCIDENT_REQUEST)
incident_tid = lab.send(incident_sid, INCIDENT_REQUEST)
tasks['incident'], _ = lab.wait(incident_tid)
db = SessionDB(run / 'hermes/state.db')
try:
    before_audit = {sid: digest(db.get_messages(sid, include_inactive=True)) for sid in (memory_sid, incident_sid)}
finally:
    db.close()


## 发现：默认搜索 user / assistant

不传 role_filter，原生发现默认搜索 user、assistant；assistant 中的工具参数也参与索引，不能误解成只搜自然语言正文。先看标题、时间与业务对象，不能把第一条命中自动当成目标事故。仅出现在 tool 回执中的内容默认不参与发现，但仍可按消息位置展开，或显式设置 role_filter='tool' 检索。原生函数不再调用模型生成摘要。

In [ ]:
db = SessionDB(run / 'hermes/state.db')
try:
    discovery = json.loads(session_search(query='rollback', limit=10, db=db))
    print(json.dumps(discovery, ensure_ascii=False, indent=2))
    hit = next((h for h in discovery.get('results', []) if h['session_id'] == incident_sid), None)
    if hit is None:
        raise RuntimeError('没有定位到事故：先核对关键词和历史范围，当前结论未知')
finally:
    db.close()


## 展开：around_message_id + window

下面保留完整返回对象，直接打印包含 tools 的窗口。用首条或末条消息编号继续移动；不把整个会话重新塞给模型。单个窗口缺少对应回执时先保留未知，再扩大可见范围。这里为便于核查遍历短教学会话，真实长会话应找到所需依据后停止。

In [ ]:
db = SessionDB(run / 'hermes/state.db')
visible = {}
try:
    for direction in ('before', 'after'):
        anchor = hit['match_message_id']
        for _ in range(200):
            view = json.loads(session_search(session_id=hit['session_id'], around_message_id=anchor, window=2, db=db))
            if not view.get('success') or not view.get('messages'):
                raise RuntimeError('无法展开：未看到的记录仍未知')
            print(json.dumps(view, ensure_ascii=False, indent=2))
            visible.update({m['id']: m for m in view['messages']})
            if not view.get('messages_' + direction):
                break
            next_anchor = view['messages'][0 if direction == 'before' else -1]['id']
            if next_anchor == anchor:
                raise RuntimeError('窗口未前进，停止而不是假装查全')
            anchor = next_anchor
        else:
            raise RuntimeError('窗口数量超过上限，结果不完整')
finally:
    db.close()
messages = [visible[k] for k in sorted(visible)]


## 配对：请求编号、执行状态与业务状态分开

assistant.tool_calls 中的 id 对应 tool 消息的 tool_call_id。缺少或重复编号时不能按相邻位置猜测；accepted 只表示受理，completed 也只说明那项操作完成。后续 describe、logs 才是业务复查，事故因果还需更多材料。

In [ ]:
# 先直接看原始编号怎样关联。不要用助手总结代替 receipt。
for message in messages:
    for call in message.get('tool_calls') or []:
        receipts = [m for m in messages if m.get('role') == 'tool' and m.get('tool_call_id') == call.get('id')]
        print({'request_message_id': message['id'], 'tool_call_id': call.get('id'), 'request': call, 'receipts': receipts})
# 共用检查函数另行拒绝缺失、重复编号，保留 unknown。
for pair in pair_messages(messages):
    print(pair['tool_call_id'], execution_state(pair), pair['receipt'])


## 同样回查 memory，而不是读取 observer

这是前台会话，不是 bg-review。原生 memory 的请求与回执应与其他工具一样出现在展开窗口。若没有同编号成功回执，当前只能说保存结果未知或失败。

In [ ]:
db = SessionDB(run / 'hermes/state.db')
try:
    found = json.loads(session_search(query='lesson13-memory', limit=10, db=db))
    memory_hit = next((h for h in found.get('results', []) if h['session_id'] == memory_sid), None)
    if memory_hit is None:
        raise RuntimeError('未找到记忆会话，不能认定已保存或从未保存')
    memory_view = json.loads(session_search(session_id=memory_sid, around_message_id=memory_hit['match_message_id'], window=5, db=db))
    print(json.dumps(memory_view, ensure_ascii=False, indent=2))
    for pair in pair_messages(memory_view.get('messages', [])):
        print(pair['tool_call_id'], pair['name'], execution_state(pair), pair['receipt'])
finally:
    db.close()


## 新会话自己核对草稿

前面的人工检查没有把会话编号或检索结果注入新会话。这里只发关键词与待核对的说法，让模型自己发现和展开；工具白名单只允许 session_search。最后检查真实工具轨迹，并另行阅读模型报告的未知范围与因果表述。

In [ ]:
audit_sid = 'audit-' + workshop.suffix
api('/sessions', {'id': audit_sid, 'title': '新会话核对两项旧事', 'history_only': True})
audit_tid = lab.send(audit_sid, AUDIT_REQUEST)
tasks['audit'], _ = lab.wait(audit_tid)
print(json.dumps(tasks['audit']['calls'], ensure_ascii=False, indent=2))


In [ ]:
try:
    result = workshop.finish(tasks, before_audit)
    print(json.dumps(result, ensure_ascii=False, indent=2))
finally:
    workshop.close()
